In [ ]:
# 1) Install Dependencies
%pip install -qU langchain langchain-google-genai langchain-chroma
%pip install -qU langchain-openrouter chromadb google-generativeai

In [8]:
import getpass
import os
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate
# Update these two lines:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [9]:
# 3) Configure API Keys and Environment
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass.getpass("Enter your Google API key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
# 4) Load Raw Menu Data
RAW_MENU = """
### Beef Biryani
- Ingredients: basmati rice, beef, onions, yogurt, tomatoes, ginger, garlic, green chilies, biryani spices (cumin, cardamom, cloves, cinnamon), saffron, herbs
- Allergens: dairy
- Dietary: non-vegetarian
- Spice Level: hot
- Cuisine Type: desi
- Specialty: yes
- Pairings: raita, salad
- Serving: single
- Price: PKR 400/plate

### Chicken Chow Mein
- Ingredients: noodles, chicken, cabbage, carrots, soy sauce, garlic, oil
- Allergens: gluten, soy
- Dietary: non-vegetarian
- Spice Level: mild
- Cuisine Type: chinese
- Specialty: yes
- Pairings: spring rolls
- Serving: single
- Price: PKR 450/plate

### Grilled Chicken Steak
- Ingredients: chicken breast, herbs, butter, black pepper
- Allergens: dairy
- Dietary: non-vegetarian
- Spice Level: mild
- Cuisine Type: continental
- Specialty: yes
- Pairings: mashed potatoes, sautéed vegetables
- Serving: single
- Price: PKR 950/plate

### Butter Chicken Pasta
- Ingredients: pasta, chicken, butter chicken sauce, cream, spices
- Allergens: dairy, gluten
- Dietary: non-vegetarian
- Spice Level: medium
- Cuisine Type: fusion
- Specialty: yes
- Pairings: garlic bread
- Serving: single
- Price: PKR 750/plate

### Plain Naan
- Ingredients: wheat flour, water, yeast, salt
- Allergens: gluten
- Dietary: vegetarian
- Spice Level: none
- Cuisine Type: desi
- Specialty: no
- Pairings: all curries
- Serving: single piece
- Price: PKR 40/piece

# Add your remaining dishes in this same format.
"""

In [ ]:
# 5) Parse Menu Text into LangChain Documents
def parse_menu(raw: str) -> list[Document]:
    """
    Splits on '### ' headings.
    Each dish becomes a Document whose:
      - page_content = full text of that dish block
      - metadata     = structured fields extracted from the block
    """
    blocks = [b.strip() for b in raw.split("###") if b.strip()]
    docs = []

    for block in blocks:
        lines = block.strip().splitlines()
        dish_name = lines[0].strip()

        page_content = f"Dish: {dish_name}\n" + "\n".join(lines[1:])

        metadata = {"dish_name": dish_name}
        field_map = {
            "Allergens": "allergens",
            "Dietary": "dietary",
            "Spice Level": "spice_level",
            "Cuisine Type": "cuisine_type",
            "Specialty": "specialty",
            "Price": "price",
            "Serving": "serving",
        }

        for line in lines[1:]:
            for label, key in field_map.items():
                if line.startswith(f"- {label}:"):
                    metadata[key] = line.split(":", 1)[1].strip()

        docs.append(Document(page_content=page_content, metadata=metadata))

    print(f"✅ Parsed {len(docs)} dishes.")
    return docs

menu_docs = parse_menu(RAW_MENU)

In [ ]:
# 6) Initialize Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=GOOGLE_API_KEY,
    task_type="RETRIEVAL_DOCUMENT",
)

In [11]:
# 7) Build and Persist Chroma Vector Store
CHROMA_DIR = "./askthemenu_chroma"

vectorstore = Chroma.from_documents(
    documents=menu_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="menu",
)

print(f"✅ ChromaDB collection has {vectorstore._collection.count()} documents.")

NameError: name 'menu_docs' is not defined

In [12]:
# 8) Create Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

NameError: name 'vectorstore' is not defined

In [ ]:
# 9) Configure OpenRouter Chat Model
llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    max_retries=2,
)

In [ ]:
# 10) Define RAG Prompt Template
SYSTEM_PROMPT = """
You are AskTheMenu, a friendly and knowledgeable AI waiter.
Your job is to help diners pick the right dish based on their
preferences, dietary needs, allergies, spice tolerance, and mood.

Use ONLY the menu context provided below. If something isn't on
the menu, say so politely. Always mention the price when recommending.
Never guess ingredients — rely strictly on the context.

Menu context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

In [ ]:
# 11) Assemble RAG Chain
def format_docs(docs: list[Document]) -> str:
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 12) Run Interactive Chat Loop
print("\n🍽️ Welcome to AskTheMenu! Type your question below.")
print("(type 'quit' to exit)\n")

while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() in ("quit", "exit", "q"):
        print("Enjoy your meal! 🙂")
        break

    response = rag_chain.invoke(user_input)
    print(f"\nAskTheMenu: {response}\n")

In [ ]:
# 13) Optional: Reload Persisted Chroma Collection
# Run this in a fresh session to skip re-embedding:
# vectorstore = Chroma(
#     persist_directory=CHROMA_DIR,
#     embedding_function=embeddings,
#     collection_name="menu",
# )

In [ ]:
# 14) Optional: Metadata-Filtered Retrievers
# Examples:
# vegetarian_retriever = vectorstore.as_retriever(
#     search_kwargs={
#         "k": 4,
#         "filter": {"dietary": "vegetarian"},
#     }
# )

# desi_retriever = vectorstore.as_retriever(
#     search_kwargs={
#         "k": 4,
#         "filter": {"cuisine_type": "desi"},
#     }
# )

In [ ]:
# 15) Optional: Quick Validation Queries
# Run after building `rag_chain`.
sample_queries = [
    "Recommend a mild non-vegetarian dish and mention the price.",
    "I have gluten allergy. What can I eat?",
    "Suggest a spicy desi item with a pairing.",
]

for q in sample_queries:
    print(f"\nQ: {q}")
    try:
        print("A:", rag_chain.invoke(q))
    except Exception as e:
        print("A: Could not run query yet:", e)